In [ ]:
from common import *

## 9. Priprema podataka

Kako ćemo u fazi imputacije trenirati modele za popunjavanje nedostajućih vrednosti, već u ovoj fazi potrebno je izvršiti feature engineering kako bismo iz dataseta izvukli maskimalnu količinu informacija.

### 9.1 Transformacija postojicih podataka

U ovom delu izvrsicemo tranformaciju podataka koji vec postoje u nasem dataframe-u na onaj nacin koji ce nasim modelima pruziti maksimalnu kolicinu informacija.

#### 9.1.1 Vremenski podaci

Za uspešnu primenu algoritama mašinskog učenja, neophodno je adekvatno pretprocesirati vremenske komponente. Ukoliko bismo datume tretirali kao klasične kategorijske promenljive i primenili 'One-Hot Encoding', to bi dovelo do eksponencijalnog rasta dimenzionalnosti skupa podataka. Sa druge strane, 'Label Encoding' unosi problem linearne progresije, gde model gubi svest o cikličnosti pri prelasku sa kraja na početak vremenskog ciklusa.

Zbog toga je logičnije primeniti metodu kružnog enkodiranja (Cyclical Encoding), transformacijom dana i meseci u tačke na dvodimenzionalnoj kružnici upotrebom sinusnih i kosinusnih funkcija. Godinu tretiramo kao nezavisnu promenljivu, ali s obzirom na to da su njene apsolutne vrednosti (2007-2017) neuporedivo veće od vrednosti sinusa i kosinusa (-1 do 1), primenićemo skaliranje godina na interval od 0 do 1. Ovo osigurava da disproporcija u magnitudama ne dovede do toga da godina dominira nad ostalim vremenskim faktorima prilikom učenja modela.

In [ ]:
data = loadData('backups/weatherAusAfter8_3.csv')
is_train = data['Date'] < SPLIT_DATE  # ponovo racunamo posle reload-a (indeks/broj redova se promenio)

data['Year_Scaled'] = (data['Year'] - data.loc[is_train, 'Year'].min()) / (data.loc[is_train, 'Year'].max() - data.loc[is_train, 'Year'].min())

data['DayOfYear_Sin'] = np.sin(2 * np.pi * data['Date'].dt.dayofyear / 365.25).round(2)
data['DayOfYear_Cos'] = np.cos(2 * np.pi * data['Date'].dt.dayofyear / 365.25).round(2)

data.drop(columns=['Year', "Month", "Day", "DayOfYear"], inplace=True)

write_log(data, "Transformacija vremenskih podataka", "weatherAusAfter9_1_1.csv")

#### 9.1.2 Podaci o vetru

U datasetu se nalazi veći broj kolona koje naznačavaju smer najjačeg vetra u određenim momentima dana, ove kolone, kao vremenski podaci, predstavljaju ordinalne kategorijske promenljive sa cikličnim redosledom. 
Ako bismo svaki smer vetar počevši od severa kodirali vrednostima počevši od jedan u smeru kazaljke na satu, dobili bismo rezultat koji će našem modelu govoriti da je severozapadni vetar najveća razlika severnome vetru, što nije tačno.
Kao i kod vremenskih podataka, da bismo rešili ovaj problem izvršićemo kružno kodiranje.

In [ ]:
wind_dir_map = {
    'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5,
    'E': 90, 'ESE': 112.5, 'SE': 135, 'SSE': 157.5,
    'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5,
    'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5
}
def wind_dir_to_sin_cos(df, col_name):
    df[col_name + '_deg'] = df[col_name].map(wind_dir_map)
    df[col_name + '_sin'] = np.sin(np.radians(df[col_name + '_deg'])).round(2)
    df[col_name + '_cos'] = np.cos(np.radians(df[col_name + '_deg'])).round(2)
    df.drop(columns=[col_name, col_name + '_deg'], inplace=True)
    return df
def convertWinds(df):
    df = wind_dir_to_sin_cos(df, 'WindDir3pm')
    df = wind_dir_to_sin_cos(df, 'WindDir9am')
    df = wind_dir_to_sin_cos(df, 'WindGustDir')
    return df
data = convertWinds(data)
write_log(data, "Transformacija podataka o vetru", "weatherAusAfter9_1_2.csv")


**Napomena o rupama u vremenskoj seriji:** u odeljku 6.4.2.1 utvrdili smo da postoje periodi kada pojedinim (ili gotovo svim) lokacijama nedostaju čitavi nizovi uzastopnih dana (najizraženije: april 2011, decembar 2012. i februar 201). Zbog toga uvodimo pomoćnu funkciju koja proverava da li razmak zaista odgovara traženom broju dana, i ako ne odgovara, vraća NaN (umesto lažne vrednosti) i takav NaN se dalje tretira identično svim ostalim nedostajućim vrednostima u odeljku 11.

### 9.2 Dodavanje novih atributa

#### 9.2.1 Diferencijalni atributi

Na osnovu domenskog znanja i logike mozemo pretpostaviti da je u meterologiji mnogo bitnija promena velicine od samih vrednosti. Na primer: ako pritisak izmedju 9 i 15 casova znatno opadne to je mnogo jaci signal da postoji verovatnoca za kisu od samih vrednost, zato uvedimo nove kolone koje ce predstavljati promenu meteroloskih velicina u intervalu od 9 do 15 casova. Takodje, mozemo dodati kolonu koja ce predstavljati dnevni raspone temperature (razliku najvise i najnize).

Imajući u vidu da i dalje imamo nedostajuce podatke i da ćemo nakon imputacije ponovo trebati da preračunamo ove kolone, kreiraćeo funkciju za računanje ovih kolona.

In [ ]:
data = calculateDiffs(data)
write_log(data, "Dodavanje diferencijalnih atributa", "weatherAusAfter9_2_1.csv")


#### 9.2.2 Pokretni proseci

Jedan od bitnih podataka kojima bismo mogli da nadomestimo veliku kolicinu nedostajucih podataka su okretni proseci, odnosno proseci određene veličine u prethodnih n dana.

In [ ]:
data = calculateRollingMeans(data)
write_log(data, "Dodavanje pokretnih proseka", "weatherAusAfter9_2_2.csv")

#### 9.2.3 Lag podaci

Jos jedan podatak koji će nam značiti u našoj budućoj analizi, pa nam je pogodno da ga imamo u svakoj vrsti, je informacija o  vrednostima odredjenih veličine pre odredjenog broja dana. Kreiraćemo funkciju za računanje ovih vrednosti.

In [ ]:
data = calculatePreviousValues(data)
write_log(data,"Dodavanje lag podataka", "weatherAusAfter9_2_3.csv")

#### 9.2.4 Funkcija za sveobuhvatno dodavanje podataka

Trenutni poziv funkcija moze kreirati vrednosti za nove atribute samo tamo gde postoje vrednosti za originalne atribute, kako cemo u nastavku vrsiti imputaciju nedostajucih podataka, javice se potreba da nakon svake faze imputacije se izvrsi ponovno kreiranje izvedenih promenjivih. 
Zbog toga cemo kreirati jednu metodu koja ce kad se pozove ponovno kreirati sve promenjljive.

#### 9.2.5 Ucitavanje podataka o stanicama

Svaka merna stanica ima različite osobine koji proazilaze iz njenih geografskih (nadmorska visina, pozicija...), ove osobine će dratično uticati na predikciju za svaku lokaciju, takođe pomoću podataka o lokaciji i nadmorskoj visini, naš model će moći zaključiti koje lokacije imaju slične osobine.
Prvo ćemo ekstrahovati sve jedinstvene nazive stanica u našem dataset-u, novodobijeni dataframe predstavlja će osnovu dalje analize podataka o stanicama.

In [ ]:
unique_stations = data['Location'].unique()

df_stanice = pd.DataFrame(unique_stations, columns=['StationName'])


##### 9.2.5.1 Geografske karakteristike

U ovom radu je realizovano automatsko preuzimanje i integracija meteoroloških podataka sa zvanične stranice Australijskog biroa za meteorologiju (BOM Climate Data Stations (https://www.bom.gov.au/climate/data/lists_by_element/stations.txt)). Osnovni cilj bio je da se strukturisani podaci o mernim stanicama sintaksički obrade i povežu sa lokalnim bazama. Primenom regularnih izraza i pozicione analize iz teksta su izdvojeni ključni parametri poput identifikatora, koordinata i nadmorske visine. Nepotpune vrednosti su automatski očišćene i mapirane radi očuvanja integriteta skupa podataka. Na kraju, ekstrahovani podaci su uspešno upareni i sačuvani u ažuriranom izlaznom fajlu stanice_updated.csv.

In [ ]:
# 1. Učitavanje izvornog tekstualnog fajla ('stations.txt')
colspecs = [(0, 7), (7, 13), (13, 54), (54, 62), (62, 70), (70, 79), (79, 89), (89, 104), (104, 108), (108, 119), (119, 128), (128, 134)]
imena_kolona = ["Site", "Dist", "Site_name", "Start", "End", "Lat", "Lon", "Source", "STA", "Height", "Bar_ht", "WMO"]

df_stations = pd.read_fwf('InputData/stations.txt', skiprows=4, colspecs=colspecs, names=imena_kolona)
df_stations['Site_name'] = df_stations['Site_name'].str.strip().str.upper()

# Formatiramo BOM ID kao string sa 6 cifara kako bismo omogućili direktno mapiranje
df_stations['Site'] = df_stations['Site'].astype(str).str.zfill(6) 
df_stations = df_stations[df_stations['Height'] != '..']

# --- Novo: Rečnik sa tačnim BOM ID kodovima za velike gradove i problematične stanice ---
# Ovo garantuje 100% tačnost za stanice koje su korišćene u "Rain in Australia" datasetu
bom_id_map = {
    'Adelaide': '023090',      # Adelaide (Kent Town)
    'Sydney': '066062',        # Sydney (Observatory Hill)
    'Melbourne': '086071',     # Melbourne (Olympic Park)
    'Brisbane': '040913',      # Brisbane
    'Perth': '009021',         # Perth Metro
    'Hobart': '094029',        # Hobart (Ellerslie Road)
    'Darwin': '014015',        # Darwin Airport
    'Canberra': '070351',      # Canberra Airport
    'SydneyAirport': '066037', # Sydney Airport AMO
    'MelbourneAirport':'086282',# Melbourne Airport
    'PerthAirport': '009014',  # Perth Airport
    'AliceSprings': '015590',  # Alice Springs Airport
    'Newcastle': '061055',     # Nobbys Signal Station
    'Wollongong': '068188',    # Illawarra Regional Airport
    'Uluru': '015643',         # Yulara Aero
    'GoldCoast': '040764',     # Gold Coast Seaway
    'Townsville': '032040',    # Townsville Aero
    'Cairns': '031011'         # Cairns Aero
}

def prilagodi_ime(ime):
    ime_string = str(ime).strip()
    
    # Hardkodovana lista da CamelCase prelom ne bi napravio greške pri pretrazi
    # Npr. 'Nhil' prelomi u 'NHIL', a BOM ga beleži kao 'NHILL'
    izuzeci = {
        'Nhil': 'NHILL',
        'PearceRAAF': 'PEARCE RAAF',
        'BadgerysCreek': 'BADGERYS CREEK',
        'CoffsHarbour': 'COFFS HARBOUR',
        'MountGinini': 'MOUNT GININI',
        'MountGambier': 'MOUNT GAMBIER',
        'NorfolkIsland': 'NORFOLK ISLAND',
        'WaggaWagga': 'WAGGA WAGGA'
    }
    
    if ime_string in izuzeci:
        return izuzeci[ime_string]
        
    ime_odvojeno = re.sub('([A-Z][a-z]+)', r' \1', re.sub('([A-Z]+)', r' \1', ime_string)).strip().upper()
    ime_odvojeno = re.sub(r'\s+', ' ', ime_odvojeno)
    return ime_odvojeno

lat_lista = []
lon_lista = []
height_lista = []
ime_kolone = 'StationName'

# 2. Logika za pretragu i uparivanje
for idx, row in df_stanice.iterrows():
    kaggle_ime = str(row[ime_kolone]).strip()
    
    # Prvo proveravamo da li imamo egzaktan BOM ID u našem rečniku (apsolutna preciznost)
    if kaggle_ime in bom_id_map:
        trazeni_id = bom_id_map[kaggle_ime]
        poklapanja = df_stations[df_stations['Site'] == trazeni_id]
    else:
        trazeno_ime = prilagodi_ime(kaggle_ime)
        
        # Striktnija pretraga: tražimo *tačno ime* ili ime koje ima razmak posle (da izbegnemo tuđa imena)
        poklapanja = df_stations[
            (df_stations['Site_name'] == trazeno_ime) | 
            (df_stations['Site_name'].str.startswith(trazeno_ime + ' ', na=False))
        ]
        
        if not poklapanja.empty:
            # Dajemo prednost zvaničnim i automatskim stanicama (AWS) ako postoji više poklapanja
            aero_stanice = poklapanja[poklapanja['Site_name'].str.contains('AIRPORT|AERO|AWS|AMO|OBSERVATORY|MO', na=False)]
            
            if not aero_stanice.empty:
                poklapanja = aero_stanice
        
    if not poklapanja.empty:
        najbolja_stanica = poklapanja.iloc[0] 
        lat_lista.append(najbolja_stanica['Lat'])
        lon_lista.append(najbolja_stanica['Lon'])
        height_lista.append(najbolja_stanica['Height'])
    else:
        lat_lista.append(None)
        lon_lista.append(None)
        height_lista.append(None)
        print(f"Upozorenje: Nije pronađena adekvatna stanica za '{kaggle_ime}'")

# Upis rezultata u DataFrame
df_stanice['Lat'] = lat_lista 
df_stanice['Lon'] = lon_lista 
df_stanice['Height'] = height_lista

df_stanice['Lat'] = df_stanice['Lat'].astype(float)
df_stanice['Lon'] = df_stanice['Lon'].astype(float)
df_stanice['Height'] = df_stanice['Height'].astype(float)

# Sačuvaj i visinu (Height) ako ti treba za prediktivne modele, često znači!
df_stanice[["StationName", "Lat", "Lon", "Height"]].to_csv("GeoPodaci/stanice.csv", index=False)

##### 9.2.5.2 Udaljenost od okeana

Poznato je da udaljenost od okeana igra ključnu ulogu u oblikovanju lokalne klime, jer velike vodene mase deluju kao ogromni termički regulatori koji ublažavaju temperaturne ekstreme i diktiraju obrasce padavina i vlažnosti vazduha. Međutim, ovaj maritimni uticaj ne slabi ravnomerno i linearno, već najdrastičnije opada u prvim desetinama kilometara od obale, zbog čega je za precizno modelovanje neophodno koristiti inverzne vrednosti udaljenosti. Na taj način matematički mnogo vernije oslikavamo taj brzi i nagli gubitak uticaja okeana kako se zalazi dublje u unutrašnjost kopna. Da bismo ovu zakonitost praktično primenili i kvantifikovali, napisana je Python skripta koja automatizuje ceo proces merenja i transformacije podataka. Skripta najpre učitava koordinate naših meteoroloških stanica, spaja ih sa detaljnom mapom australijske obale i sve prebacuje u metrički sistem kako bi proračuni u metrima i kilometrima bili potpuno tačni. Zatim algoritam, grubo rečeno, iz svake stanice "baca zrake" u četiri osnovna pravca sveta (sever, jug, istok i zapad) i pronalazi tačku u kojoj te zamišljene linije "udaraju" u obalu. Kada se na taj način izračuna fizičko rastojanje, kod automatski invertuje dobijene kilometre i kreira nove indikatore. Kao rezultat, dobijamo novu bazu u kojoj svaka stanica ima jasan, numerički izražen uticaj okeana sa sve četiri strane sveta, spremnu za dalje statističke analize i obradu

In [ ]:
import ssl
if GEO_LIBS_AVAILABLE:
    ssl._create_default_https_context = ssl._create_unverified_context
    # 1. Kreiranje geografskih tacaka (GeoDataFrame) na osnovu Lon i Lat kolona
    geometrija = [Point(xy) for xy in zip(df_stanice['Lon'], df_stanice['Lat'])]
    gdf_stanice = gpd.GeoDataFrame(df_stanice, geometry=geometrija)

    # Postavljamo originalni koordinatni sistem (WGS84 - stepeni)
    gdf_stanice.set_crs(epsg=4326, inplace=True)

    # 2. Pribavljanje geometrije obale (koristimo cartopy)
    shpfilename = shpreader.natural_earth(resolution='10m',
                                          category='physical',
                                          name='coastline')

    obale_kolekcija = list(shpreader.Reader(shpfilename).geometries())
    # Izdvajamo samo delove obale koji su blizu Australije
    australija_obale = [geom for geom in obale_kolekcija if geom.bounds[0] > 110 and geom.bounds[2] < 160 and geom.bounds[1] > -50 and geom.bounds[3] < 0]

    if len(australija_obale) > 1:
        kombinovana_obala = linemerge(australija_obale)
    else:
        kombinovana_obala = australija_obale[0]

    # 3. Prebacivanje u metricki sistem (EPSG:3577) za precizno racunanje udaljenosti
    gdf_stanice = gdf_stanice.to_crs(epsg=3577)

    projekcija_wgs84 = pyproj.CRS('EPSG:4326')
    projekcija_aus = pyproj.CRS('EPSG:3577')
    projektor = pyproj.Transformer.from_crs(projekcija_wgs84, projekcija_aus, always_xy=True).transform

    obala_metricka = transform(projektor, kombinovana_obala)

    # 4. Racunanje udaljenosti za cetiri strane sveta i inverznih vrednosti
    def izracunaj_usmerenu_udaljenost(tacka, pravac):
        x, y = tacka.x, tacka.y
        duzina_zraka = 5000000  # 5000 km u metrima (dovoljno da presece obalu iz bilo koje tacke)

        if pravac == 'Sever':
            krajnja_tacka = (x, y + duzina_zraka)
        elif pravac == 'Jug':
            krajnja_tacka = (x, y - duzina_zraka)
        elif pravac == 'Istok':
            krajnja_tacka = (x + duzina_zraka, y)
        elif pravac == 'Zapad':
            krajnja_tacka = (x - duzina_zraka, y)

        zrak = LineString([(x, y), krajnja_tacka])
        presek = zrak.intersection(obala_metricka)

        if presek.is_empty:
            return 5000.0
        else:
            return tacka.distance(presek) / 1000

    df_stanice['Dist_Sever_km'] = gdf_stanice['geometry'].apply(lambda pt: izracunaj_usmerenu_udaljenost(pt, 'Sever'))
    df_stanice['Dist_Jug_km'] = gdf_stanice['geometry'].apply(lambda pt: izracunaj_usmerenu_udaljenost(pt, 'Jug'))
    df_stanice['Dist_Istok_km'] = gdf_stanice['geometry'].apply(lambda pt: izracunaj_usmerenu_udaljenost(pt, 'Istok'))
    df_stanice['Dist_Zapad_km'] = gdf_stanice['geometry'].apply(lambda pt: izracunaj_usmerenu_udaljenost(pt, 'Zapad'))

    df_stanice['Inv_Dist_Sever'] = 1 / (df_stanice['Dist_Sever_km'] + 1)
    df_stanice['Inv_Dist_Jug'] = 1 / (df_stanice['Dist_Jug_km'] + 1)
    df_stanice['Inv_Dist_Istok'] = 1 / (df_stanice['Dist_Istok_km'] + 1)
    df_stanice['Inv_Dist_Zapad'] = 1 / (df_stanice['Dist_Zapad_km'] + 1)

    df_rezultat = pd.DataFrame(df_stanice.drop(columns='geometry', errors='ignore'))
else:
    print("[info] Preskacem racunanje udaljenosti od okeana (geopandas/cartopy nedostupni),"
          " ucitavam vec sacuvane vrednosti...")
    _cache = pd.read_csv("backups/weatherAusAfter9_2_5_4.csv")
    _lookup_cols = ['Location', 'Inv_Dist_Sever', 'Inv_Dist_Jug', 'Inv_Dist_Istok', 'Inv_Dist_Zapad']
    _lookup = _cache[_lookup_cols].drop_duplicates(subset=['Location']).rename(columns={'Location': 'StationName'})
    df_rezultat = df_stanice.merge(_lookup, on='StationName', how='left')


##### 9.2.5.3 Klimatska zona

Radi preciznije analize prostornih i meteoroloških karakteristika terena, mernje stanice iz dataseta klasifikovane su prema zvaničnoj tipologiji Australijskog meteorološkog zavoda (BOM). Ovaj sistem, baziran na modifikovanoj Kepenovoj klasifikaciji, deli teritoriju na osnovu višegodišnjeg režima temperatura i rasporeda padavina.
U cilju integracije ovih podataka u analitički model, definisan je rečnik preslikavanja climate_mapping koji svaku stanicu pridružuje odgovarajućoj klimatskoj grupi:
Uvođenjem ove kategorizacije obezbeđuje se da algoritmi za mašinsko učenje uzmu u obzir regionalne klimatske specifičnosti prilikom obrade podataka o padavinama.

<i>Podaci o klimatskim zonama su izvađeni sa sajta: https://www.bom.gov.au/resources/learn-and-explore/climate-knowledge-centre/australian-climate-zones</i>

In [ ]:
climate_mapping = {
    'Darwin': 'Tropical', 'Cairns': 'Tropical', 'Katherine': 'Tropical',
    'Brisbane': 'Subtropical', 'GoldCoast': 'Subtropical', 'CoffsHarbour': 'Subtropical',
    'Newcastle': 'Subtropical', 'NorahHead': 'Subtropical', 'Sydney': 'Subtropical',
    'SydneyAirport': 'Subtropical', 'Wollongong': 'Subtropical', 'Williamtown': 'Subtropical',
    'BadgerysCreek': 'Subtropical', 'Penrith': 'Subtropical', 'Richmond': 'Subtropical',
    'Perth': 'Subtropical', 'PerthAirport': 'Subtropical', 'PearceRAAF': 'Subtropical',
    'NorfolkIsland': 'Subtropical',
    'AliceSprings': 'Desert', 'Woomera': 'Desert',
    'Cobar': 'Grassland', 'Moree': 'Grassland', 'Mildura': 'Grassland', 'SalmonGums': 'Grassland',
    'Melbourne': 'Temperate', 'MelbourneAirport': 'Temperate', 'Ballarat': 'Temperate',
    'Bendigo': 'Temperate', 'Watsonia': 'Temperate', 'Adelaide': 'Temperate',
    'MountGambier': 'Temperate', 'Nuriootpa': 'Temperate', 'Canberra': 'Temperate',
    'Tuggeranong': 'Temperate', 'MountGinini': 'Temperate', 'Hobart': 'Temperate',
    'Launceston': 'Temperate', 'Albury': 'Temperate', 'WaggaWagga': 'Temperate',
    'Albany': 'Temperate', 'Witchcliffe': 'Temperate', 'Walpole': 'Temperate',
    'Dartmoor': 'Temperate', 'Portland': 'Temperate', 'Sale': 'Temperate', "Uluru": "Desert", "Townsville": "Tropical", "Nhil": "Temperate"
}

df_rezultat['ClimateGroup'] = df_rezultat['StationName'].map(climate_mapping)

##### 9.2.5.4 Dodavanja podataka o stanici u originalni dataframe

Sve relevantne informacije o metereoloskim stanicama nalaze se u novodobijenom dataframe-u, konacno, mozemo povezati originalni dataframe sa dataframe-om o metereoloskim stanicama.

In [ ]:
def pripremi_spojen_df(df_main, df_elev):
    df = df_main.copy()
    elev_dict = df_elev.set_index('StationName')['Height'].to_dict()
    lat_dict = df_elev.set_index("StationName")["Lat"].to_dict()
    lon_dict = df_elev.set_index("StationName")["Lon"].to_dict()
    distanca_okean = {}
    for x in ["Inv_Dist_Sever", "Inv_Dist_Jug", "Inv_Dist_Istok","Inv_Dist_Zapad"]:
        df_elev[x] = df_elev[x].round(4)
        distanca_okean[x] = df_elev.set_index('StationName')[x].to_dict()
    klima_dict = df_elev.set_index('StationName')['ClimateGroup'].to_dict()
    
    df['Nadmorska visina (m)'] = df['Location'].map(elev_dict)
    df['Nadmorska visina (m)'] = df['Nadmorska visina (m)'].astype(float)

    for x in ["Inv_Dist_Sever", "Inv_Dist_Jug", "Inv_Dist_Istok","Inv_Dist_Zapad"]:
        df[x] = df['Location'].map(distanca_okean[x])
    df['klima'] = df['Location'].map(klima_dict)
    df["Lat"] = df["Location"].map(lat_dict)
    df["Lon"] = df["Location"].map(lon_dict)

    df['Lat'] = df['Lat'].astype(float)
    df['Lon'] = df['Lon'].astype(float)

    df = pd.get_dummies(df, columns=['klima'], drop_first=True, dtype=int)
    
    return df
data = pripremi_spojen_df(data, df_rezultat)
write_log(data, "Ucitavanje podataka o stanicana", "weatherAusAfter9_2_5_4.csv")

#### 9.2.7 Dodavanje podataka o tacki rose

Tačka rose predstavlja temperaturu na koju vazduh mora da se ohladi, pri konstantnom pritisku, kako bi postao potpuno zasićen vodenom parom. Kada se temperatura vazduha izjednači sa tačkom rose, vodena para počinje da se kondenzuje, a relativna vlažnost vazduha dostiže 100%.
Potrebne podatke preuzeli smo sa zvanične Iowa Environmental Mesonet ASOS platforme, koja sadrži detaljne arhive meteoroloških merenja. Kako bismo osigurali tačnost, bilo je ključno uskladiti preuzete podatke sa lokalnim vremenskim zonama svih posmatranih lokacija širom Australije. Pomoću Python koda, precizno smo izdvojili vrednosti tačke rose zabeležene tačno u 9 ujutru i 3 popodne i integrisali ih u naš glavni dataset.

U nastavku je prikazan kod koji ilustruje proces izdvajanja, prilagođavanja vremenskih zona i spajanja novih podataka sa glavnim skupom.

In [ ]:
def obradi_tacku_rose(ulazni_fajl, izlazni_fajl):
    df = pd.read_csv(ulazni_fajl)

    df = df.rename(columns={
        'station': 'Lokacija',
        'valid': 'Vreme_UTC',
        'lat': 'Lat',
        'lon': 'Lon',
        'dwpc': 'TackaRose'
    })

    df['Vreme_UTC'] = pd.to_datetime(df['Vreme_UTC'], utc=True)

    tf = TimezoneFinder()
    jedinstvene_lok = df[['Lokacija', 'Lat', 'Lon']].drop_duplicates()

    def nadji_zonu(lat, lon):
        zona = tf.timezone_at(lng=lon, lat=lat)
        return zona if zona else 'UTC'

    jedinstvene_lok['Zona'] = jedinstvene_lok.apply(lambda row: nadji_zonu(row['Lat'], row['Lon']), axis=1)

    df = df.merge(jedinstvene_lok[['Lokacija', 'Zona']], on='Lokacija', how='left')

    df['Vreme_Lokalno'] = pd.NaT
    for zona, grupa in df.groupby('Zona'):
        lokalna_zona = pytz.timezone(zona)
        df.loc[grupa.index, 'Vreme_Lokalno'] = grupa['Vreme_UTC'].dt.tz_convert(lokalna_zona).dt.tz_localize(None)

    df['Datum'] = df['Vreme_Lokalno'].dt.date

    df['Idealno_9am'] = pd.to_datetime(df['Datum'].astype(str) + ' 09:00:00')
    df['Idealno_3pm'] = pd.to_datetime(df['Datum'].astype(str) + ' 15:00:00')

    df['Razlika_9am'] = (df['Vreme_Lokalno'] - df['Idealno_9am']).abs()
    df['Razlika_3pm'] = (df['Vreme_Lokalno'] - df['Idealno_3pm']).abs()

    max_tolerancija = pd.Timedelta(hours=2)

    df_9am = df[df['Razlika_9am'] <= max_tolerancija].copy()
    df_9am = df_9am.sort_values('Razlika_9am').drop_duplicates(subset=['Lokacija', 'Datum'])
    df_9am = df_9am.rename(columns={'TackaRose': 'TackaRose9am', 'Vreme_UTC': 'univerzalnoVremeZa9am'})

    df_3pm = df[df['Razlika_3pm'] <= max_tolerancija].copy()
    df_3pm = df_3pm.sort_values('Razlika_3pm').drop_duplicates(subset=['Lokacija', 'Datum'])
    df_3pm = df_3pm.rename(columns={'TackaRose': 'TackaRose3pm', 'Vreme_UTC': 'univerzalnoVremeZa3pm'})

    izlaz_df = pd.merge(
        df_9am[['Lokacija', 'Datum', 'TackaRose9am', 'univerzalnoVremeZa9am']],
        df_3pm[['Lokacija', 'Datum', 'TackaRose3pm', 'univerzalnoVremeZa3pm']],
        on=['Lokacija', 'Datum'],
        how='outer'
    )

    izlaz_df = izlaz_df.sort_values(['Lokacija', 'Datum'])

    izlaz_df.to_csv(izlazni_fajl, index=False)
    print(f"Gotovo! Podaci su sačuvani u: {izlazni_fajl}")

if TIMEZONEFINDER_AVAILABLE and not os.path.exists('reports/tacka_rose.csv'):
    obradi_tacku_rose('InputData/asos.csv', 'reports/tacka_rose.csv')
else:
    print("[info] reports/tacka_rose.csv vec postoji (ili timezonefinder nedostupan) -"
          " preskacem ponovno racunanje tacke rose.")

Sada cemo povezati novodobijeni dataframe sa originalnim.

In [ ]:
df_tacka_rose = pd.read_csv("reports/tacka_rose.csv")

mapiranje_lokacija = {
    'YABA': 'Albany', 'YAYE': 'Uluru', 'YBAS': 'AliceSprings', 
    'YBBN': 'Brisbane', 'YBCS': 'Cairns', 'YBTL': 'Townsville', 
    'YCBA': 'Cobar', 'YCFS': 'CoffsHarbour', 'YMAY': 'Albury', 
    'YMEN': 'Melbourne', 'YMHB': 'Hobart', 'YMIA': 'Mildura', 
    'YMOR': 'Moree', 'YMTG': 'MountGambier', 'YPAD': 'Adelaide', 
    'YPDN': 'Darwin', 'YPJT': 'Perth', 'YPTN': 'Katherine', 
    'YPWR': 'Woomera', 'YSCB': 'Canberra', 'YSNF': 'NorfolkIsland', 
    'YSWG': 'WaggaWagga', 'YWLM': 'Williamtown'
}

df_tacka_rose['StationName'] = df_tacka_rose['Lokacija'].map(mapiranje_lokacija)

data['Date'] = pd.to_datetime(data['Date'])
df_tacka_rose['Datum'] = pd.to_datetime(df_tacka_rose['Datum'])
data = pd.merge(
    data,
    df_tacka_rose,
    left_on=['Location', 'Date'],
    right_on=['StationName', 'Datum'],
    how='left'
)

data = data.drop(columns=['StationName', 'Datum', 'Lokacija', "univerzalnoVremeZa9am", "univerzalnoVremeZa3pm"])
write_log(data, "Dodavanje podataka o tacki rose", "weatherAusAfter9_2_5.csv")


#### 9.2.8 Dodavanje indikatora o nedostajucim podacima

U okviru pretprocesiranja skupa weatherAUS.csv, sproveden je postupak kreiranja indikatorskih kolona koje eksplicitno beleže izostanak podataka. Za svaku postojeću kolonu napravljena je njena dvojnica u kojoj vrednost 1 označava nedostatak podatka, dok 0 označava njegovo prisustvo. Značaj ove tehnike ogleda se u činjenici da izostanak vrednosti u realnim sistemima često nije slučajan i može nositi važnu prediktivnu informaciju. Ukoliko bismo nedostajuće vrednosti samo veštački popunili, algoritam bi izgubio uvid u to koji podaci su originalno izmereni. Zadržavanjem ovih informacija kroz indikatore, modelima mašinskog učenja omogućavamo da prepoznaju šablone u izostanku merenja, što znatno povećava preciznost i robusnost finalnog modela. Postupak je programski rešen u Python biblioteci pandas korišćenjem funkcija za detekciju praznina, koje logičke vrednosti uspešno transformišu u numerički binarni format. Novonastale kolone su zatim obeležene jasnim sufiksom i spojene sa originalnim datasetom, čime je on proširen bez gubitka početnih informacija.

In [ ]:
indikatori = data.isna().astype(int).add_suffix('_missing')
indikatori = indikatori.loc[:, indikatori.any()]
df_prosiren = pd.concat([data, indikatori], axis=1)
write_log(df_prosiren, "Dodavanje indikatora o nedostajucim podacima", "weatherAusAfter9_2_6.csv")